In [64]:
# ==========================================
# Import Required Libraries
# ==========================================

import pandas as pd
import numpy as np

In [65]:
# ==========================================
# Load Dataset
# ==========================================

df = pd.read_csv("../data/processed/cleaned_sensor_data.csv")

In [66]:
df.head()

,sensor_id,timestamp,location_id,pH,turbidity,TDS,DO,flow_rate,pressure,alert_flag,contamination_target,leak_target
0,SNS-LOCA01-001,2025-05-01 00:00:00+00:00,ZONE-A,7.30,1.01,232.71,7.60,12.79,347.65,normal,0,0
1,SNS-LOCA02-002,2025-05-01 00:00:00+00:00,ZONE-A,7.22,1.83,229.58,8.14,13.55,353.83,normal,0,0
2,SNS-LOCB01-001,2025-05-01 00:00:00+00:00,ZONE-B,7.11,1.21,212.95,8.20,12.62,368.69,normal,0,0
3,SNS-LOCB02-002,2025-05-01 00:00:00+00:00,ZONE-B,7.08,1.06,231.53,8.57,13.40,356.69,normal,0,0
4,SNS-LOCC01-001,2025-05-01 00:00:00+00:00,ZONE-C,7.41,1.46,198.66,7.79,13.86,338.45,normal,0,0


In [67]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17280 entries, 0 to 17279
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   sensor_id             17280 non-null  str    
 1   timestamp             17280 non-null  str    
 2   location_id           17280 non-null  str    
 3   pH                    17280 non-null  float64
 4   turbidity             17280 non-null  float64
 5   TDS                   17280 non-null  float64
 6   DO                    17280 non-null  float64
 7   flow_rate             17280 non-null  float64
 8   pressure              17280 non-null  float64
 9   alert_flag            17280 non-null  str    
 10  contamination_target  17280 non-null  int64  
 11  leak_target           17280 non-null  int64  
dtypes: float64(6), int64(2), str(4)
memory usage: 1.6 MB


In [68]:
df.describe()

,pH,turbidity,TDS,DO,flow_rate,pressure,contamination_target,leak_target
count,17280.000000,17280.000000,17280.000000,17280.000000,17280.000000,17280.000000,17280.000000,17280.000000
mean,7.199883,1.506184,221.133760,8.004267,15.429931,349.627622,0.005208,0.004572
std,0.160445,0.440502,28.689149,0.401943,1.919588,9.716227,0.071983,0.067462
min,5.510000,0.060000,146.150000,6.430000,9.440000,216.670000,0.000000,0.000000
25%,7.100000,1.230000,209.600000,7.730000,14.080000,344.470000,0.000000,0.000000
50%,7.200000,1.500000,219.835000,8.000000,15.380000,349.910000,0.000000,0.000000
75%,7.300000,1.770000,230.100000,8.280000,16.700000,355.350000,0.000000,0.000000
max,7.780000,11.750000,791.940000,9.910000,32.770000,380.640000,1.000000,1.000000


In [69]:
df.isnull().sum()

sensor_id               0
timestamp               0
location_id             0
pH                      0
turbidity               0
TDS                     0
DO                      0
flow_rate               0
pressure                0
alert_flag              0
contamination_target    0
leak_target             0
dtype: int64

In [70]:
df.duplicated().sum()

np.int64(0)

In [71]:
# Feature 1: Convert Timestamp
# The timestamp is currently text.

df["timestamp"] = pd.to_datetime(df["timestamp"])

In [72]:
# Feature 2: Hour of Day

df["hour"] = df["timestamp"].dt.hour

In [73]:
# Feature 3: Day

df["day"] = df["timestamp"].dt.day

In [74]:
# Feature 4: Month

df["month"] = df["timestamp"].dt.month

In [75]:
# Feature 5: Day of Week

df["day_of_week"] = df["timestamp"].dt.dayofweek

In [76]:
# Feature 6: Weekend

df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

# Feature Engineering: Safe pH Indicator

The **pH** value measures the acidity or alkalinity of water and is one of the most important indicators of water quality. According to widely accepted drinking water standards, a pH value between **6.5 and 8.5** is generally considered safe for drinking water.

Instead of using only the continuous pH values, we create a new binary feature called **`is_ph_normal`**. This feature indicates whether the pH reading falls within the recommended safe range.

- **1** → pH is within the normal range (6.5–8.5)
- **0** → pH is outside the normal range

This process is an example of **feature engineering**, where we transform an existing feature into a more meaningful representation that may help the machine learning model better distinguish between normal and abnormal water conditions.

In [77]:
# ==========================================================
# Feature Engineering: Safe pH Indicator
# ==========================================================
# pH measures the acidity or alkalinity of water.
# According to drinking water guidelines, the ideal
# pH range is between 6.5 and 8.5.
#
# Here, we create a new binary feature:
#
# 1 -> pH is within the safe range
# 0 -> pH is outside the safe range
#
# Binary features are often easier for machine learning
# models to interpret and can improve classification
# performance by explicitly highlighting important conditions.
# ==========================================================

df["is_ph_normal"] = (
    (df["pH"] >= 6.5) &   # Check if pH is at least 6.5
    (df["pH"] <= 8.5)     # Check if pH is at most 8.5
).astype(int)             # Convert True/False into 1/0

# Feature Engineering: High Turbidity Indicator

**Turbidity** measures the cloudiness or haziness of water caused by suspended particles such as silt, clay, organic matter, and microorganisms. It is an important water quality parameter because high turbidity can reduce water clarity and may indicate contamination.

According to commonly accepted drinking water guidelines, turbidity values **below 5 NTU (Nephelometric Turbidity Units)** are generally considered acceptable. Values above this threshold may suggest poor water quality and require further investigation.

In this step, we create a new binary feature called **`high_turbidity`** to indicate whether the turbidity value exceeds the recommended threshold.

- **1** → High turbidity (greater than 5 NTU)
- **0** → Normal turbidity (5 NTU or below)

Creating this binary feature simplifies the information for the machine learning model, allowing it to more easily identify conditions that may be associated with contamination or abnormal water quality.

In [78]:
# ==========================================================
# Feature Engineering: High Turbidity Indicator
# ==========================================================
# Turbidity measures how cloudy or unclear the water is.
# Higher turbidity values often indicate suspended particles
# such as dirt, microorganisms, or other contaminants.
#
# According to commonly accepted drinking water guidelines,
# turbidity values greater than 5 NTU are considered high.
#
# Here, we create a binary feature:
#
# 1 -> High turbidity (greater than 5 NTU)
# 0 -> Normal turbidity (5 NTU or below)
#
# This feature makes it easier for machine learning models
# to identify potential contamination events.
# ==========================================================

df["high_turbidity"] = (
    df["turbidity"] > 5      # Check whether turbidity exceeds 5 NTU
).astype(int)                # Convert True/False into 1/0

# Feature Engineering: High Total Dissolved Solids (TDS) Indicator

**Total Dissolved Solids (TDS)** is the measure of the total concentration of dissolved substances in water, including minerals, salts, metals, and organic matter. It is one of the key parameters used to assess water quality.

According to commonly accepted drinking water guidelines, a **TDS value below 500 mg/L** is generally considered acceptable for drinking water. Values exceeding this limit may indicate excessive dissolved solids, which can negatively affect the taste, quality, and overall suitability of water.

In this step, we create a new binary feature called **`high_tds`** to identify whether a water sample has a TDS value above the recommended threshold.

- **1** → High TDS (greater than 500 mg/L)
- **0** → Acceptable TDS (500 mg/L or below)

By converting a continuous numerical measurement into a binary indicator, we provide the machine learning model with an additional feature that explicitly represents whether the water exceeds the recommended TDS limit. This can improve the model's ability to identify poor water quality and predict abnormal conditions.

In [79]:
# ==========================================================
# Feature Engineering: High Total Dissolved Solids (TDS)
# ==========================================================
# TDS (Total Dissolved Solids) represents the total amount
# of dissolved minerals, salts, metals, and organic matter
# present in water.
#
# Drinking water with TDS values below 500 mg/L is generally
# considered acceptable according to commonly accepted
# drinking water guidelines.
#
# Here, we create a binary feature:
#
# 1 -> TDS is greater than 500 mg/L (High TDS)
# 0 -> TDS is 500 mg/L or below (Acceptable TDS)
#
# This engineered feature provides the machine learning
# model with a clear indication of whether the water sample
# exceeds the recommended TDS threshold.
# ==========================================================

df["high_tds"] = (
    df["TDS"] > 500      # Check if TDS exceeds 500 mg/L
).astype(int)            # Convert True/False values into 1/0

# Feature Engineering: Low Dissolved Oxygen (DO) Indicator

**Dissolved Oxygen (DO)** measures the amount of oxygen dissolved in water and is an important indicator of water quality and the health of aquatic ecosystems. Adequate dissolved oxygen is essential for the survival of fish, aquatic plants, and other microorganisms.

In general, a **DO value below 5 mg/L** is considered low and may indicate poor water quality, pollution, or unfavorable environmental conditions. Such conditions can stress aquatic life and may also signal abnormalities in the water distribution system.

In this step, we create a new binary feature called **`low_do`** that identifies whether the dissolved oxygen level is below the recommended threshold.

- **1** → Low Dissolved Oxygen (less than 5 mg/L)
- **0** → Normal Dissolved Oxygen (5 mg/L or above)

This binary feature provides the machine learning model with a clear indicator of potentially poor water quality, making it easier to identify abnormal conditions and improve alert prediction.

In [80]:
# ==========================================================
# Feature Engineering: Low Dissolved Oxygen (DO) Indicator
# ==========================================================
# Dissolved Oxygen (DO) measures the amount of oxygen
# present in water and is an important indicator of water
# quality and aquatic ecosystem health.
#
# A DO value below 5 mg/L is generally considered low and
# may indicate poor water quality or possible contamination.
#
# Here, we create a binary feature:
#
# 1 -> DO is less than 5 mg/L (Low Dissolved Oxygen)
# 0 -> DO is 5 mg/L or above (Normal Dissolved Oxygen)
#
# This engineered feature helps the machine learning model
# quickly identify water samples that may have insufficient
# oxygen levels and could be associated with abnormal
# water quality conditions.
# ==========================================================

df["low_do"] = (
    df["DO"] < 5          # Check if Dissolved Oxygen is below 5 mg/L
).astype(int)             # Convert True/False into 1/0

# Feature Engineering: Flow-to-Pressure Ratio

In a water distribution system, **flow rate** and **pressure** are closely related hydraulic parameters. Individually, they provide useful information about the system, but analyzing them together can reveal operational conditions that may not be obvious from either measurement alone.

In this step, we create a new feature called **`flow_pressure_ratio`**, which represents the relationship between the amount of water flowing through the pipeline and the corresponding pressure.

A significantly **high ratio** may indicate that a large amount of water is flowing while the pressure is relatively low, which could suggest abnormal operating conditions such as leakage, excessive water demand, or pipe damage. Conversely, a **low ratio** may indicate restricted flow or unusually high pressure.

This is an example of an **interaction feature**, where two existing variables are combined to create a new feature that captures their relationship. Interaction features often provide machine learning models with more informative patterns than using the original variables independently.

In [81]:
# ==========================================================
# Feature Engineering: Flow-to-Pressure Ratio
# ==========================================================
# Flow rate and pressure are two important hydraulic
# parameters in a water distribution network.
#
# Instead of treating them as completely independent
# variables, we combine them into a new feature called
# 'flow_pressure_ratio'.
#
# This interaction feature describes how much water is
# flowing relative to the pressure in the pipeline.
#
# A higher ratio may indicate:
# • High water flow with relatively low pressure
# • Possible leakage
# • Abnormal water demand
# • Pipe damage or pressure loss
#
# We add 1 to the pressure value to avoid division by
# zero in case any pressure reading is 0.
# ==========================================================

df["flow_pressure_ratio"] = (
    df["flow_rate"] /          # Amount of water flowing through the pipe
    (df["pressure"] + 1)       # Add 1 to prevent division by zero
)

# Feature Engineering: Pressure Category

**Pressure** is an important hydraulic parameter in a water distribution system. It represents the force with which water flows through pipelines and plays a significant role in maintaining efficient water supply.

Although pressure is originally recorded as a continuous numerical value, it is sometimes beneficial to group the readings into meaningful categories such as **Low**, **Medium**, and **High**. This process is known as **feature binning** or **discretization**.

In this step, the pressure values are divided into three categories:

- **Low:** 0 – 30
- **Medium:** 30 – 60
- **High:** 60 – 100

Creating pressure categories allows the machine learning model to recognize broader operating conditions rather than focusing only on exact numerical values. This can improve pattern recognition, especially when pressure variations are associated with leakage, abnormal water demand, or system faults.

> **Note:** These pressure ranges are assumed for this project. In a real-world production system, the category boundaries should be determined using domain knowledge, engineering standards, or statistical analysis of the dataset.

In [82]:
# ==========================================================
# Feature Engineering: Pressure Category
# ==========================================================
# Pressure is a continuous numerical feature measured by
# sensors in the water distribution system.
#
# Here, we convert the continuous pressure values into
# three meaningful categories:
#
# Low    :   0 - 30
# Medium :  30 - 60
# High   :  60 - 100
#
# This process is called Feature Binning (Discretization).
#
# Instead of working only with exact numerical values,
# the machine learning model can also learn broader
# operating conditions of the water network.
#
# NOTE:
# These threshold values are assumed for this project.
# In real-world applications, pressure ranges should be
# selected based on engineering standards or the
# statistical distribution of the dataset.
# ==========================================================

df["pressure_category"] = pd.cut(
    df["pressure"],              # Numerical pressure values
    bins=[0, 30, 60, 100],       # Pressure intervals (bin edges)
    labels=["Low", "Medium", "High"]   # Category labels for each interval
)

# Feature Engineering: Flow Rate Category

**Flow rate** represents the volume of water passing through a pipeline over a given period of time. It is one of the most important hydraulic parameters in a water distribution system because it reflects water demand, consumption patterns, and the overall operating condition of the network.

Instead of using only the continuous numerical flow rate values, we convert them into three meaningful categories: **Low**, **Medium**, and **High**. This process is known as **feature binning (discretization)**.

The flow rate values are divided into the following ranges:

- **Low:** 0 – 50
- **Medium:** 50 – 100
- **High:** Above 100

Grouping continuous values into categories allows the machine learning model to learn general operational patterns more effectively. For example, unusually high flow rates may indicate excessive water demand, pipeline leakage, or abnormal system behavior, while very low flow rates may indicate restricted supply or low consumption.

> **Note:** The threshold values used in this project are assumed for demonstration purposes. In real-world water distribution systems, these ranges should be determined using engineering standards, historical sensor data, or statistical analysis.

In [83]:
# ==========================================================
# Feature Engineering: Flow Rate Category
# ==========================================================
# Flow rate measures the quantity of water flowing through
# the pipeline over a period of time.
#
# Instead of using only continuous numerical values, we
# divide the flow rate into three categories:
#
# Low    :   0 - 50
# Medium :  50 - 100
# High   : Above 100
#
# This process is called Feature Binning (Discretization).
#
# Creating categorical features helps the machine learning
# model recognize broader operating conditions such as
# normal demand, low demand, or unusually high water flow,
# which may be associated with leakage or abnormal usage.
#
# NOTE:
# These threshold values are assumed for this project.
# In real-world applications, they should be selected
# using domain knowledge or statistical analysis.
# ==========================================================

df["flow_category"] = pd.cut(
    df["flow_rate"],                 # Numerical flow rate values
    bins=[0, 50, 100, 1000],         # Flow rate intervals (bin edges)
    labels=["Low", "Medium", "High"] # Category labels for each interval
)

# Feature Engineering: Water Quality Score

Water quality is determined by multiple physical and chemical parameters rather than a single sensor reading. Parameters such as **pH**, **turbidity**, **Total Dissolved Solids (TDS)**, and **Dissolved Oxygen (DO)** each describe different aspects of water quality.

Instead of allowing the machine learning model to learn from these measurements independently, we create a new engineered feature called **`water_quality_score`**. This feature combines multiple sensor readings into a single numerical score that represents the overall condition of the water.

The score is calculated using a weighted combination of the sensor values:

- **pH (30%)** – Measures the acidity or alkalinity of water.
- **Turbidity (25%)** – Lower turbidity generally indicates cleaner water.
- **TDS (20%)** – Lower dissolved solids generally indicate better water quality.
- **DO (25%)** – Higher dissolved oxygen generally indicates healthier water.

The weights used in this calculation are chosen for demonstration purposes and are intended to create an informative feature for machine learning. They do **not** represent an official Water Quality Index (WQI) used by environmental agencies.

This engineered feature helps summarize multiple water quality measurements into a single indicator that may improve the model's ability to distinguish between normal and abnormal water conditions.

In [84]:
# ==========================================================
# Feature Engineering: Water Quality Score
# ==========================================================
# Water quality depends on several sensor measurements,
# including pH, turbidity, Total Dissolved Solids (TDS),
# and Dissolved Oxygen (DO).
#
# Instead of treating these parameters independently,
# we combine them into a single engineered feature called
# 'water_quality_score'.
#
# The score is calculated using a weighted combination:
#
# • pH          -> 30%
# • Turbidity   -> 25%
# • TDS         -> 20%
# • DO          -> 25%
#
# Lower turbidity and lower TDS generally indicate
# better water quality, so they are transformed before
# being included in the score.
#
# NOTE:
# This is a custom feature engineered for this machine
# learning project. It is NOT an official Water Quality
# Index (WQI) formula.
# ==========================================================

df["water_quality_score"] = (
    df["pH"] * 0.30
    + (10 - df["turbidity"]) * 0.25
    + ((1000 - df["TDS"]) / 100) * 0.20
    + df["DO"] * 0.25
)

# Feature Engineering: One-Hot Encoding

The features **`pressure_category`** and **`flow_category`** were created in the previous steps using feature binning. These columns contain categorical values such as **"Low"**, **"Medium"**, and **"High"**.

However, most machine learning algorithms cannot directly process text (categorical) values. Therefore, these categorical features must be converted into numerical representations before they can be used for model training.

In this step, **One-Hot Encoding** is applied using the `pd.get_dummies()` function. This technique converts each category into a separate binary (0 or 1) column.

For example, instead of storing a single column with three text values:

| pressure_category |
|-------------------|
| Low |
| Medium |
| High |

One-Hot Encoding transforms it into:

| pressure_category_Medium | pressure_category_High |
|--------------------------|-------------------------|
| 0 | 0 |
| 1 | 0 |
| 0 | 1 |

The **Low** category is automatically represented when both columns contain **0**, because `drop_first=True` removes the first category to avoid redundancy.

This encoding allows the machine learning model to process categorical information efficiently while avoiding unintended numerical relationships between categories.

In [85]:
# ==========================================================
# Feature Engineering: One-Hot Encoding
# ==========================================================
# The columns 'pressure_category' and 'flow_category'
# contain categorical (text) values such as:
#
# Low
# Medium
# High
#
# Machine learning algorithms cannot directly process
# text values. Therefore, we convert these categories
# into binary (0/1) numerical columns using
# One-Hot Encoding.
#
# Example:
#
# Pressure Category
# -----------------
# Low
# Medium
# High
#
# becomes
#
# pressure_category_Medium
# pressure_category_High
#
# Low    -> 0 0
# Medium -> 1 0
# High   -> 0 1
#
# We use drop_first=True to remove the first category
# ("Low") and avoid multicollinearity (also called the
# Dummy Variable Trap), where one encoded column can be
# perfectly predicted from the others.
# ==========================================================

df = pd.get_dummies(
    df,
    columns=[
        "pressure_category",
        "flow_category"
    ],
    drop_first=True
)

In [86]:
# Checking the Result

df.head()

,sensor_id,timestamp,location_id,pH,turbidity,TDS,DO,flow_rate,pressure,alert_flag,...,is_ph_normal,high_turbidity,high_tds,low_do,flow_pressure_ratio,water_quality_score,pressure_category_Medium,pressure_category_High,flow_category_Medium,flow_category_High
0,SNS-LOCA01-001,2025-05-01 00:00:00+00:00,ZONE-A,7.30,1.01,232.71,7.60,12.79,347.65,normal,...,1,0,0,0,0.036684,7.87208,False,False,False,False
1,SNS-LOCA02-002,2025-05-01 00:00:00+00:00,ZONE-A,7.22,1.83,229.58,8.14,13.55,353.83,normal,...,1,0,0,0,0.038187,7.78434,False,False,False,False
2,SNS-LOCB01-001,2025-05-01 00:00:00+00:00,ZONE-B,7.11,1.21,212.95,8.20,12.62,368.69,normal,...,1,0,0,0,0.034137,7.95460,False,False,False,False
3,SNS-LOCB02-002,2025-05-01 00:00:00+00:00,ZONE-B,7.08,1.06,231.53,8.57,13.40,356.69,normal,...,1,0,0,0,0.037463,8.03844,False,False,False,False
4,SNS-LOCC01-001,2025-05-01 00:00:00+00:00,ZONE-C,7.41,1.46,198.66,7.79,13.86,338.45,normal,...,1,0,0,0,0.040831,7.90818,False,False,False,False


In [87]:
# Save the Dataset  

df.to_csv(
    "../data/features/water_quality_features.csv",
    index=False
)

In [88]:
# Just For Verification

print(df.shape)

df.info()

df.head()

(17280, 27)
<class 'pandas.DataFrame'>
RangeIndex: 17280 entries, 0 to 17279
Data columns (total 27 columns):
 #   Column                    Non-Null Count  Dtype              
---  ------                    --------------  -----              
 0   sensor_id                 17280 non-null  str                
 1   timestamp                 17280 non-null  datetime64[us, UTC]
 2   location_id               17280 non-null  str                
 3   pH                        17280 non-null  float64            
 4   turbidity                 17280 non-null  float64            
 5   TDS                       17280 non-null  float64            
 6   DO                        17280 non-null  float64            
 7   flow_rate                 17280 non-null  float64            
 8   pressure                  17280 non-null  float64            
 9   alert_flag                17280 non-null  str                
 10  contamination_target      17280 non-null  int64              
 11  leak_target   

,sensor_id,timestamp,location_id,pH,turbidity,TDS,DO,flow_rate,pressure,alert_flag,...,is_ph_normal,high_turbidity,high_tds,low_do,flow_pressure_ratio,water_quality_score,pressure_category_Medium,pressure_category_High,flow_category_Medium,flow_category_High
0,SNS-LOCA01-001,2025-05-01 00:00:00+00:00,ZONE-A,7.30,1.01,232.71,7.60,12.79,347.65,normal,...,1,0,0,0,0.036684,7.87208,False,False,False,False
1,SNS-LOCA02-002,2025-05-01 00:00:00+00:00,ZONE-A,7.22,1.83,229.58,8.14,13.55,353.83,normal,...,1,0,0,0,0.038187,7.78434,False,False,False,False
2,SNS-LOCB01-001,2025-05-01 00:00:00+00:00,ZONE-B,7.11,1.21,212.95,8.20,12.62,368.69,normal,...,1,0,0,0,0.034137,7.95460,False,False,False,False
3,SNS-LOCB02-002,2025-05-01 00:00:00+00:00,ZONE-B,7.08,1.06,231.53,8.57,13.40,356.69,normal,...,1,0,0,0,0.037463,8.03844,False,False,False,False
4,SNS-LOCC01-001,2025-05-01 00:00:00+00:00,ZONE-C,7.41,1.46,198.66,7.79,13.86,338.45,normal,...,1,0,0,0,0.040831,7.90818,False,False,False,False
